## Simple Value Object

In [20]:
class Value:
    def __init__(self, data):
        self.data = data

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        out = Value(self.data + other.data)
        return out

    def __mul__(self, other):
        out = Value(self.data*other.data)
        return out
    

In [21]:
a = Value(2)
b = Value(3)

In [24]:
# a.__add__(b)
a+b

Value(data=5)

In [26]:
# a.__add__(b)

In [23]:
# a.__mul__(b)
a*b

Value(data=6)

In [3]:
a.data

2

## Remembering the children of the Value Object

In [106]:
class Value:
    def __init__(self, data, _children = (), _op = '', label = ''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
        self.label = label
        self.grad = 0.0

    def __repr__(self):
        # return f"Value(data={self.data}, children={self._prev})"
        return f"Value(data={self.data})"

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), '+')
        self.grad += out.grad * other.data
        other.grad += out.grad * self.data
        return out

    def __mul__(self, other):
        out = Value(self.data*other.data, (self, other), '*')
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(t, (self, ), 'tanh')
        return out

In [123]:
a = Value(2, label = 'a')
b = Value(3, label = 'b')

In [101]:
c = a + b; c.label = 'c'
d = Value(4, label = 'd')
e = d * c; e.label = 'e'

In [102]:
e

Value(data=20)

In [103]:
e.grad = 1

In [104]:
d.grad

0.0

In [91]:
c._prev

{Value(data=2), Value(data=3)}

In [94]:
# from graphviz import Digraph

# def trace(root):
#   # builds a set of all nodes and edges in a graph
#   nodes, edges = set(), set()
#   def build(v):
#     if v not in nodes:
#       nodes.add(v)
#       for child in v._prev:
#         edges.add((child, v))
#         build(child)
#   build(root)
#   return nodes, edges

# def draw_dot(root):
#   dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
  
#   nodes, edges = trace(root)
#   for n in nodes:
#     uid = str(id(n))
#     # for any value in the graph, create a rectangular ('record') node for it
#     dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
#     if n._op:
#       # if this value is a result of some operation, create an op node for it
#       dot.node(name = uid + n._op, label = n._op)
#       # and connect this node to it
#       dot.edge(uid + n._op, uid)

#   for n1, n2 in edges:
#     # connect n1 to the op node of n2
#     dot.edge(str(id(n1)), str(id(n2)) + n2._op)

#   return dot

In [95]:
# draw_dot(c)

In [105]:
1

1

## Implementing the backward Pass

In [131]:
import math

In [173]:
class Value:
    def __init__(self, data, _children = (), _op = '', label = ''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
        self.label = label
        self.grad = 0.0
        self._backward = lambda : None

    def __repr__(self):
        # return f"Value(data={self.data}, children={self._prev})"
        return f"Value(data={self.data})"

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += out.grad * 1
            other.grad += out.grad * 1

        out._backward = _backward
        return out

    def __mul__(self, other):
        out = Value(self.data*other.data, (self, other), '*')

        def _backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data

        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += out.grad*(1 - t**2)

        out._backward = _backward
        return out

    def backward(self):
        self.grad = 1
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        for node in reversed(topo):
            node._backward()
        # topo

In [186]:
a = Value(-1, label = 'a')
b = Value(3, label = 'b')
c = a + b; c.label = 'c'
d = Value(2, label = 'd')
e = d * c; e.label = 'e'
o = e.tanh(); o.label = 'o'

In [187]:
e

Value(data=4)

In [188]:
o

Value(data=0.999329299739067)

In [189]:
o.grad, e.grad, d.grad

(0.0, 0.0, 0.0)

In [190]:
o.backward()

In [191]:
o.grad, e.grad, d.grad, c.grad

(1, 0.0013409506830258655, 0.002681901366051731, 0.002681901366051731)

In [154]:
print(e.grad)

0.07065082485316443


In [157]:
topo = []
visited = set()
def build_topo(v):
    if v not in visited:
        visited.add(v)
        for child in v._prev:
            build_topo(child)
        topo.append(v)
build_topo(o)
topo

[Value(data=2),
 Value(data=3),
 Value(data=-2),
 Value(data=1),
 Value(data=2),
 Value(data=0.9640275800758169)]